# Car Specs EDA

In [1]:
import pandas as pd

In [2]:
cars = pd.read_csv('/Users/eric5805/Downloads/csv/data/final/cars/car_stats.csv')
features = pd.read_csv('/Users/eric5805/Downloads/csv/data/final/cars/car_features.csv')
print(f'cars: {cars.shape}  |  features: {features.shape}')

cars: (27902, 30)  |  features: (27902, 43)


In [3]:
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27902 entries, 0 to 27901
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   review_id            26088 non-null  object 
 1   car_id               27902 non-null  object 
 2   make                 27902 non-null  object 
 3   model                27902 non-null  object 
 4   year                 27902 non-null  int64  
 5   bodytype             24988 non-null  object 
 6   doors                24872 non-null  float64
 7   trim                 27902 non-null  object 
 8   price                24811 non-null  float64
 9   mpg_city             25519 non-null  float64
 10  mpg_hwy              25520 non-null  float64
 11  mpg_comb             25791 non-null  float64
 12  Fuel Type            27902 non-null  object 
 13  hp                   24451 non-null  float64
 14  Engine               26500 non-null  object 
 15  torque_lbft          24303 non-null 

In [4]:
cars.isnull().sum().sort_values(ascending=False)

0 - 60                 20749
cargo_cuft              9041
Width with mirrors      7696
Turning Diameter        6152
Front Leg Room          5113
Front Shoulder Room     5108
Recommended Fuel        4934
Front Head Room         4803
curb_weight_lbs         4767
torque_lbft             3599
Fuel Capacity           3562
hp                      3451
Transmission Type       3351
Overall Length          3219
Wheel Base              3132
price                   3091
doors                   3030
bodytype                2914
mpg_city                2383
mpg_hwy                 2382
mpg_comb                2111
review_id               1814
Engine                  1402
Drivetrain                 4
car_id                     0
Fuel Type                  0
year                       0
model                      0
make                       0
trim                       0
dtype: int64

## Flags

In [5]:
cars['is_ev'] = cars['Fuel Type'].str.contains('Electric', case=False, na=False)

In [6]:
luxury = [
    'Acura', 'Alfa-Romeo', 'Audi', 'Bentley', 'BMW', 'Cadillac', 'Ferrari',
    'Genesis', 'Infiniti', 'Jaguar', 'Lamborghini', 'Land-Rover', 'Lexus',
    'Lincoln', 'Lucid', 'Maserati', 'McLaren Automotive', 'Mercedes-Benz',
    'Polestar', 'Porsche', 'Rivian', 'Rolls-Royce', 'Tesla', 'Volvo'
]
cars['luxury'] = cars['make'].isin(luxury).astype(int)

## Imputation

### Doors

In [7]:
doors_missing = (
    cars
    .groupby('bodytype')['doors']
    .apply(lambda x: x.isna().sum())
)
print(doors_missing[doors_missing > 0])

bodytype
Hatchback     2
Minivan       8
Pickup       95
SUV          11
Name: doors, dtype: int64


In [8]:
four_doors = 'Double Cab|Quad Cab|Super Cab|Crew Cab|Flex Titanium|Extended Cab|Suburban|Yukon XL|Sienna|Sequoia|Sedona|RLX|MDX|ZDX|'
two_doors  = 'King Cab|XtraCab|Access Cab|Regular Cab|Club Cab|Clubman John Cooper'
three_doors = 'Accent Blue'

cars.loc[
    (cars['trim'].str.contains(four_doors, case=False, na=False)) & (cars['doors'].isna()),
    'doors'
] = 4
cars.loc[
    (cars['trim'].str.contains(two_doors, case=False, na=False)) & (cars['doors'].isna()),
    'doors'
] = 2
cars.loc[
    (cars['trim'].str.contains(three_doors, case=False, na=False)) & (cars['doors'].isna()),
    'doors'
] = 3

print(f'Remaining door nulls: {cars["doors"].isna().sum()}')

Remaining door nulls: 0


### Price

In [9]:
cars['price'] = cars.groupby(['make', 'bodytype', 'year'])['price'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Remaining price nulls: {cars["price"].isna().sum()}')

Remaining price nulls: 2914


### Recommended Fuel

In [10]:
SPECIAL_TECH_PAT = r'(?:turbo|bi.?turbo|twin.?turbo|supercharge|\bv8\b|\bv10\b|\bv12\b)'
has_tech = (
    cars['trim'].str.contains(SPECIAL_TECH_PAT, case=False, na=False)
    | cars['Engine'].str.contains(SPECIAL_TECH_PAT, case=False, na=False)
)

null_m = cars['Recommended Fuel'].isna()

# 1. Electric / Hydrogen
cars.loc[null_m & cars['Fuel Type'].isin(['Electric', 'Hydrogen Fuel Cell']),
         'Recommended Fuel'] = 'Electric'
null_m = cars['Recommended Fuel'].isna()

# 2. Diesel
cars.loc[null_m & cars['Fuel Type'].eq('Diesel'), 'Recommended Fuel'] = 'Diesel'
null_m = cars['Recommended Fuel'].isna()

# 3. Natural Gas
cars.loc[null_m & cars['Fuel Type'].eq('Natural Gas'), 'Recommended Fuel'] = 'CNG'
null_m = cars['Recommended Fuel'].isna()

# 4. Luxury + special tech -> Premium
cars.loc[null_m & (cars['luxury'] == 1) & has_tech, 'Recommended Fuel'] = 'Premium'
null_m = cars['Recommended Fuel'].isna()

# 5. Remaining -> Regular
cars.loc[null_m, 'Recommended Fuel'] = 'Regular'

print(f'Remaining nulls: {cars["Recommended Fuel"].isna().sum()}')
cars.groupby('Fuel Type')['Recommended Fuel'].value_counts(dropna=False)

Remaining nulls: 0


Fuel Type           Recommended Fuel
Diesel              Diesel               1327
Electric            Electric              904
Flex-Fuel           Regular              2148
                    Premium                61
                    E85                     3
Gas                 Regular             12744
                    Premium              8496
Hybrid              Regular              1344
                    Premium               479
Hydrogen Fuel Cell  Electric               36
Natural Gas         CNG                    17
Plug-in Hybrid      Regular               322
                    Premium                21
Name: count, dtype: int64

## Drop Incomplete Rows

Drop rows missing any of the core analytical columns: `review_id`, `Engine`, `Transmission Type`, `mpg_city`, `mpg_hwy`, `mpg_comb`.

In [11]:
DROP_COLS = ['review_id', 'Engine', 'Transmission Type', 'mpg_city', 'mpg_hwy', 'mpg_comb']
before = len(cars)
cars = cars.dropna(subset=DROP_COLS).reset_index(drop=True)
print(f'Dropped {before - len(cars):,} rows ({(before - len(cars)) / before:.1%}) → {len(cars):,} remaining')

Dropped 5,890 rows (21.1%) → 22,012 remaining


### Horsepower

Median by `(Engine, Fuel Type)` — the engine string encodes displacement and induction type, making it a precise key.

### Curb Weight

Median by `(bodytype, Fuel Type, make)`, fallback `(bodytype, Fuel Type)`.

### Price (fallback)

Already filled by `(make, bodytype, year)` above; apply fallback tiers for rows still null after the drop.

In [12]:
# Fallback: (make, bodytype) then (make,)
cars['price'] = cars.groupby(['make', 'bodytype'])['price'].transform(
    lambda x: x.fillna(x.median())
)
cars['price'] = cars.groupby('make')['price'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Remaining price nulls: {cars["price"].isna().sum()}')

Remaining price nulls: 0


### Dimensions

Columns: `Wheel Base`, `Overall Length`, `Width with mirrors`, `Turning Diameter`,
`Front Head Room`, `Front Leg Room`, `Front Shoulder Room`, `Fuel Capacity`, `cargo_cuft`.

**Three-level cascade (mean, not median — variance is low within group):**
1. `make + model + year` — same nameplate, same model year (R² 0.92–0.99)
2. `make + model` — across years (R² 0.91–0.94)
3. `make + bodytype` — same maker + body class (last resort)

`cargo_cuft` for Pickups is **structurally absent** (traditional trucks don't report
cargo volume in ft³; only EV trucks like Rivian R1T / Tesla Cybertruck do).
Pickup rows are excluded from cargo imputation and left as NaN.

~1,772 EPA-registry rows (FFV / payload-variant trims, all `bodytype=NaN`) will
fall through all three levels and remain NaN — imputing phantom dimensions onto
fuel-economy regulatory entries would be misleading.

In [13]:
# Dimensions: three-level cascade fill (make+model+year → make+model → make+bodytype)
# cargo_cuft Pickup rows are excluded — structurally absent, not missing.

DIM_COLS = [
    'Wheel Base', 'Overall Length', 'Width with mirrors', 'Turning Diameter',
    'Front Head Room', 'Front Leg Room', 'Front Shoulder Room',
    'Fuel Capacity', 'cargo_cuft',
]

FILL_GROUPS = [
    ['make', 'model', 'year'],   # most precise: same nameplate + year
    ['make', 'model'],           # across years
    ['make', 'bodytype'],        # same maker + body class
]

before = cars[DIM_COLS].isnull().sum()

for col in DIM_COLS:
    for grp in FILL_GROUPS:
        still_null = cars[col].isnull()
        if not still_null.any():
            break
        # Exclude Pickups from cargo_cuft imputation (structurally absent)
        if col == 'cargo_cuft':
            eligible = cars['bodytype'] != 'Pickup'
            group_mean = cars.loc[eligible].groupby(grp)[col].transform('mean')
            cars.loc[still_null & eligible, col] = group_mean[still_null & eligible]
        else:
            group_mean = cars.groupby(grp)[col].transform('mean')
            cars[col] = cars[col].fillna(group_mean)

after = cars[DIM_COLS].isnull().sum()
summary = (
    before.rename('before')
    .to_frame()
    .assign(after=after, filled=before - after)
)
print(summary.to_string())


                     before  after  filled
Wheel Base              125      0     125
Overall Length          218      0     218
Width with mirrors     4064    158    3906
Turning Diameter       2779     87    2692
Front Head Room        1547     43    1504
Front Leg Room         1865     43    1822
Front Shoulder Room    1848     43    1805
Fuel Capacity           154      0     154
cargo_cuft             4867   3053    1814


### Null Summary

In [14]:
remaining = cars.isnull().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)
print(f'Total rows: {len(cars):,}')
print(f'\nRemaining nulls:')
print(remaining)

Total rows: 22,012

Remaining nulls:
0 - 60                 15446
cargo_cuft              3053
curb_weight_lbs         1479
torque_lbft              413
hp                       317
Width with mirrors       158
Turning Diameter          87
Front Head Room           43
Front Leg Room            43
Front Shoulder Room       43
dtype: int64


In [15]:
# Ordinal 0=Not Available / 1=Optional / 2=Standard → scale to [0, 0.5, 1]
feat_cols_to_norm = [c for c in features.columns if c not in ('car_id', 'review_id')]
features_norm = features.copy()
features_norm[feat_cols_to_norm] = features_norm[feat_cols_to_norm].div(2)

print(f'features_norm shape: {features_norm.shape}')
print('Unique values after scaling:', sorted(features_norm[feat_cols_to_norm].stack().unique()))

features_norm shape: (27902, 43)
Unique values after scaling: [0.0, 0.5, 1.0]


### Filter to Core Dataset

Keep only rows whose `car_id` exists in `cars_norm` (22,012 rows). Also drop `review_id` — it's a join key, not a feature.

In [16]:
# Align to the 22,012 rows kept in cars_norm
features_norm = features_norm[features_norm['car_id'].isin(cars_norm['car_id'])].reset_index(drop=True)
features_norm.drop(columns='review_id', inplace=True)

print(f'features_norm shape after filter: {features_norm.shape}')
print(f'Remaining nulls: {features_norm.isnull().sum().sum()}')

NameError: name 'cars_norm' is not defined

### Impute Missing Feature Flags

Fill NaN with the **mode** (most frequent value) per column. Since the dataset is skewed toward "Standard" (1.0), this is a conservative, data-driven fill.

In [17]:
print('=== cars_norm ===')
print(f'  shape : {cars_norm.shape}')
print(f'  nulls : {cars_norm.isnull().sum().sum()}')
print(f'  dtypes: {cars_norm.dtypes.value_counts().to_dict()}')
print()
print('=== features_norm ===')
feat_cols_norm = [c for c in features_norm.columns if c not in ('car_id', 'review_id')]
print(f'  shape : {features_norm.shape}')
print(f'  nulls : {features_norm[feat_cols_norm].isnull().sum().sum()}')
print()
print('Value counts in features_norm:')
print(features_norm[feat_cols_norm].stack().value_counts().sort_index())

=== cars_norm ===


NameError: name 'cars_norm' is not defined

## Feature Availability

In [18]:
# Filter features to the 22,012 core rows (matches filtered cars DataFrame)
feat_filtered = features[features['car_id'].isin(cars['car_id'])].reset_index(drop=True)
feat_data_cols = [c for c in feat_filtered.columns if c not in ('car_id', 'review_id')]

availability = (
    feat_filtered[feat_data_cols]
    .apply(lambda col: pd.Series({
        'standard_%':  (col == 2).mean() * 100,
        'optional_%':  (col == 1).mean() * 100,
        'not_avail_%': (col == 0).mean() * 100,
        'missing_%':   col.isna().mean() * 100,
    }))
    .T
    .round(1)
    .sort_values('standard_%', ascending=False)
)
availability

,standard_%,optional_%,not_avail_%,missing_%
Power Windows,92.4,4.1,1.6,1.9
Child Seat Anchors,92.1,0.2,5.2,2.5
Tilt Steering Wheel,91.3,1.8,1.3,5.6
Cruise Control,86.8,6.2,4.5,2.4
Alloy Wheels,84.2,10.2,5.4,0.3
Stability Control,82.8,1.3,1.4,14.5
Traction Control,80.9,2.2,2.6,14.3
Remote Keyless Entry,75.9,4.1,2.6,17.4
Cup Holder Count,75.7,0.3,2.3,21.7
Child Door Locks,74.7,0.0,20.9,4.3


In [19]:
merged_feat = feat_filtered.merge(cars[['car_id', 'luxury']], on='car_id')
feat_by_luxury = (
    merged_feat.groupby('luxury')[feat_data_cols]
    .apply(lambda df: (df == 2).mean() * 100)
    .T
    .round(1)
)
feat_by_luxury.columns = ['non_luxury_std%', 'luxury_std%']
feat_by_luxury['luxury_advantage'] = (feat_by_luxury['luxury_std%'] - feat_by_luxury['non_luxury_std%']).round(1)
feat_by_luxury.sort_values('luxury_advantage', ascending=False)

,non_luxury_std%,luxury_std%,luxury_advantage
Dual Power Front Seats,26.6,87.8,61.2
Rain Sensing Windshield Wipers,13.9,71.4,57.5
Leather Seats,33.6,76.2,42.6
Power Driver's Seat,50.5,91.5,41.0
Power Folding Exterior Mirrors,9.6,48.6,39.0
Navigation System,21.0,55.0,34.0
Premium Radio,18.8,50.4,31.6
Tilt/Telescoping Steering Wheel,61.7,87.8,26.1
Heated Mirrors,56.2,79.9,23.7
Alarm System,54.3,76.7,22.4


## EV Share by Year

> **Note:** EVs are absent from the filtered `cars` dataset (22,012 rows) because electric vehicles have no MPG values and were removed by the `dropna` step. This section loads the full raw `car_stats.csv` to show the actual EV trend across model years.

In [20]:
cars_full = pd.read_csv('/Users/eric5805/Downloads/csv/data/final/cars/car_stats.csv')
cars_full['is_ev'] = cars_full['Fuel Type'].str.contains('Electric', case=False, na=False)

ev_by_year = (
    cars_full.groupby('year')['is_ev']
    .agg(ev_trims='sum', total='count')
)
ev_by_year['ev_pct'] = (ev_by_year['ev_trims'] / ev_by_year['total'] * 100).round(1)
ev_by_year[ev_by_year['total'] >= 5]

,ev_trims,total,ev_pct
year,,,
2000,4,25,16.0
2001,5,386,1.3
2002,2,422,0.5
2003,1,582,0.2
2004,0,676,0.0
2005,0,742,0.0
2006,0,759,0.0
2007,0,791,0.0
2008,1,856,0.1


In [21]:
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22012 entries, 0 to 22011
Data columns (total 32 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   review_id            22012 non-null  object 
 1   car_id               22012 non-null  object 
 2   make                 22012 non-null  object 
 3   model                22012 non-null  object 
 4   year                 22012 non-null  int64  
 5   bodytype             22012 non-null  object 
 6   doors                22012 non-null  float64
 7   trim                 22012 non-null  object 
 8   price                22012 non-null  float64
 9   mpg_city             22012 non-null  float64
 10  mpg_hwy              22012 non-null  float64
 11  mpg_comb             22012 non-null  float64
 12  Fuel Type            22012 non-null  object 
 13  hp                   21695 non-null  float64
 14  Engine               22012 non-null  object 
 15  torque_lbft          21599 non-null 

In [22]:
cars.to_csv('/Users/eric5805/Downloads/csv/data/processed/cars_stats_final.csv', index=False)